In [75]:
from nlp_utils import *
import pandas as pd
import numpy as np

# open data, convert to dataframes

In [76]:
df_train = open_data('./data/liar-plus/train2.tsv')
df_test = open_data('./data/liar-plus/test2.tsv')
df_val = open_data('./data/liar-plus/val2.tsv')
df_train.head()

,id,label,statement,subject,speaker,speaker_job_title,state_info,party_affiliation,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_on_fire_counts,context,justification,party_category,word_count,topic_list
0,2635.json,false,Says the Annies List political group supports ...,abortion,dwayne-bohac,State representative,Texas,republican,0.0,1.0,0.0,0.0,0.0,a mailer,That's a premise that he fails to back up. Ann...,right-leaning,11,[abortion]
1,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0.0,0.0,1.0,1.0,0.0,a floor speech.,"Surovell said the decline of coal ""started whe...",left-leaning,24,"[energy, history, job-accomplishments]"
2,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70.0,71.0,160.0,163.0,9.0,Denver,Obama said he would have voted against the ame...,left-leaning,19,[foreign-policy]
3,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,NaN,NaN,none,7.0,19.0,3.0,5.0,44.0,a news release,The release may have a point that Mikulskis co...,other,12,[health-care]
4,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist,NaN,Florida,democrat,15.0,9.0,20.0,19.0,2.0,an interview on CNN,"Crist said that the economic ""turnaround start...",left-leaning,10,"[economy, jobs]"


In [77]:
for df in [df_train, df_test, df_val]:
    df["statement"] = df["statement"].astype(str)
    df['statement'].dropna(inplace=True)
    df["statement"].apply(clean_text)

# Factuality Factor 1: BERT Transformer

In [78]:
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from datasets import Dataset, DatasetDict

for df in [df_train, df_val, df_test]:
    # drop rows where statement is missing; reset index
    df.dropna(subset=['statement'], inplace=True)
    df.reset_index(drop=True, inplace=True)
    df['statement'] = df['statement'].astype(str)

hf_train = Dataset.from_pandas(df_train[['statement','label']])
hf_val   = Dataset.from_pandas(df_val[['statement','label']])
hf_test  = Dataset.from_pandas(df_test[['statement','label']])
dataset = DatasetDict({'train': hf_train, 'validation': hf_val, 'test': hf_test})

le = LabelEncoder()
all_labels = np.concatenate([dataset[k]['label'] for k in dataset])
le.fit(all_labels)
num_labels = len(le.classes_)
print("Classes:", list(le.classes_))

def encode_label(example):
    example['label'] = int(le.transform([example['label']])[0])
    return example

dataset = dataset.map(encode_label)

MODEL_NAME = "bert-base-uncased"   # swap if you prefer another checkpoint
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(examples):
    return tokenizer(examples['statement'],
                     truncation=True,
                     padding=False)   # we'll use data collator to pad per-batch

dataset = dataset.map(preprocess, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

def compute_metrics(pred):
    labels = pred.label_ids
    preds  = np.argmax(pred.predictions, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro'),
        'precision_macro': precision_score(labels, preds, average='macro', zero_division=0),
        'recall_macro': recall_score(labels, preds, average='macro', zero_division=0)
    }

output_dir = "./bert_finetuned_liar_plus"
training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    save_total_limit=2,
    logging_dir=f"{output_dir}/logs",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

metrics = trainer.evaluate(dataset['test'])
print(metrics)

trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

Classes: ['barely-true', 'false', 'half-true', 'mostly-true', 'pants-fire', 'true', None]


Map: 100%|██████████| 1267/1267 [00:00<00:00, 33968.81 examples/s]
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: f657be42-bc95-491b-b788-8be05f1175b5)')' thrown while requesting HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
Map: 100%|██████████| 1267/1267 [00:00<00:00, 61069.23 examples/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/var/folders/pd/jy2tczp11tvgftzcy6kn7txh0000gn/T/ipykernel_73937/2853934088.py:74: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/Library/Frameworks/Python.framework

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Macro,Recall Macro
1,3.909100,5.456667,0.204829,0.056669,0.034138,0.166667
2,3.358900,4.356074,0.131620,0.038770,0.021937,0.166667
3,2.960000,3.416257,0.184579,0.051940,0.030763,0.166667
4,2.354400,2.140417,0.195483,0.054506,0.032580,0.166667
5,2.106600,1.766611,0.195483,0.054506,0.032580,0.166667


Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG15XFamilyCommandBuffer: 0x6238d95a0>
    label = <none> 
    device = <AGXG15SDevice: 0x139eddc00>
        name = Apple M3 Pro 
    commandQueue = <AGXG15XFamilyCommandQueue: 0x36804dc00>
        label = <none> 
        device = <AGXG15SDevice: 0x139eddc00>
            name = Apple M3 Pro 
    retainedReferences = 1
Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG15XFamilyCommandBuffer: 0x8a3834b80>
    label = <none> 
    device = <AGXG15SDevice: 0x139eddc00>
        name = Apple M3 Pro 
    commandQueue = <AGXG15XFamilyCommandQueue: 0x36804dc00>
        label = <none> 
        device =

{'eval_loss': 5.375620365142822, 'eval_accuracy': 0.19652722967640096, 'eval_f1_macro': 0.05474934036939314, 'eval_precision_macro': 0.03275453827940016, 'eval_recall_macro': 0.16666666666666666, 'eval_runtime': 5.4453, 'eval_samples_per_second': 232.676, 'eval_steps_per_second': 7.346, 'epoch': 5.0}


('./bert_finetuned_liar_plus/tokenizer_config.json',
 './bert_finetuned_liar_plus/special_tokens_map.json',
 './bert_finetuned_liar_plus/vocab.txt',
 './bert_finetuned_liar_plus/added_tokens.json',
 './bert_finetuned_liar_plus/tokenizer.json')

Consider using sentence transformer instead of BERT

In [79]:
# df_train['statement'] = df_train['statement'].astype(str)

# train_encodings = tokenizer(
#     df_train['statement'].tolist(),
#     truncation=True,
#     padding=True,  # pad to longest in batch
#     return_tensors='pt'
# )

# model.eval()
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model.to(device)

# input_ids = train_encodings['input_ids'].to(device)
# attention_mask = train_encodings['attention_mask'].to(device)

# batch_size = 32
# all_probs = []

# for i in range(0, len(df_train), batch_size):
#     batch_ids = input_ids[i:i+batch_size]
#     batch_mask = attention_mask[i:i+batch_size]
#     with torch.no_grad():
#         batch_outputs = model(batch_ids, attention_mask=batch_mask)
#         batch_probs = torch.softmax(batch_outputs.logits, dim=1)
#         all_probs.append(batch_probs.cpu())

# probs = torch.cat(all_probs, dim=0)

# df_train['bert_pred_class'] = torch.argmax(probs, dim=1).numpy()

# num_labels = probs.shape[1]
# for i in range(num_labels):
#     df_train[f'bert_prob_class_{i}'] = probs[:, i].numpy()

In [80]:
batch_size = 32
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.eval()
model.to(device)

def bert_pred(df_in):
    df = df_in.copy()
    # Ensure statements are strings
    df['statement'] = df['statement'].astype(str)

    # Tokenize this specific df
    encodings = tokenizer(
        df['statement'].tolist(),
        truncation=True,
        padding=True,
        return_tensors='pt'
    )

    input_ids = encodings['input_ids'].to(device)
    attention_mask = encodings['attention_mask'].to(device)

    # Batch prediction
    all_probs = []
    for i in range(0, len(df), batch_size):
        batch_ids = input_ids[i:i+batch_size]
        batch_mask = attention_mask[i:i+batch_size]
        with torch.no_grad():
            batch_outputs = model(batch_ids, attention_mask=batch_mask)
            batch_probs = torch.softmax(batch_outputs.logits, dim=1)
            all_probs.append(batch_probs.cpu())

    probs = torch.cat(all_probs, dim=0)

    # Predicted class
    df['bert_pred_class'] = torch.argmax(probs, dim=1).numpy()

    # Probabilities per class
    num_labels = probs.shape[1]
    for j in range(num_labels):
        df[f'bert_prob_class_{j}'] = probs[:, j].numpy()
    return df

for df in [df_train, df_val, df_test]:
    df = bert_pred(df)

# Factuality Factor 2: Spam

In [81]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd

spam_model_name = "mrm8488/bert-tiny-finetuned-sms-spam-detection"
spam_tokenizer = AutoTokenizer.from_pretrained(spam_model_name)
spam_model = AutoModelForSequenceClassification.from_pretrained(spam_model_name)
spam_model.eval()

def get_spam_scores(text_list, batch_size=16):
    scores = []
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        inputs = spam_tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding="max_length",
            max_length=512
        )
        with torch.no_grad():
            outputs = spam_model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)
            scores.extend(probs[:, 1].tolist())
    return scores

In [82]:
df_train['spam_score'] = get_spam_scores(df_train['statement'].tolist())
df_test['spam_score'] = get_spam_scores(df_test['statement'].tolist())
df_val['spam_score'] = get_spam_scores(df_val['statement'].tolist())

In [83]:
df_train.head()

,id,label,statement,subject,speaker,speaker_job_title,state_info,party_affiliation,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_on_fire_counts,context,justification,party_category,word_count,topic_list,spam_score
0,2635.json,false,Says the Annies List political group supports ...,abortion,dwayne-bohac,State representative,Texas,republican,0.0,1.0,0.0,0.0,0.0,a mailer,That's a premise that he fails to back up. Ann...,right-leaning,11,[abortion],0.121970
1,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0.0,0.0,1.0,1.0,0.0,a floor speech.,"Surovell said the decline of coal ""started whe...",left-leaning,24,"[energy, history, job-accomplishments]",0.063852
2,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70.0,71.0,160.0,163.0,9.0,Denver,Obama said he would have voted against the ame...,left-leaning,19,[foreign-policy],0.067691
3,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,NaN,NaN,none,7.0,19.0,3.0,5.0,44.0,a news release,The release may have a point that Mikulskis co...,other,12,[health-care],0.121808
4,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist,NaN,Florida,democrat,15.0,9.0,20.0,19.0,2.0,an interview on CNN,"Crist said that the economic ""turnaround start...",left-leaning,10,"[economy, jobs]",0.062488


In [84]:
for df in [df_train, df_test, df_val]:
    df['spam_class'] = df['spam_score'].apply(lambda x: 1 if x > 0.67 else 0.5 if x > 0.33 else 0)
    df['spam_class'] = df['spam_class'].astype('category')

# Factuality Factor 3: Political Bias (Ryan)

In [85]:
import spacy
# spacy.cli.download("en_core_web_md")
datum = df_train.iloc[0]
nlp = spacy.load("en_core_web_md")
doc = nlp(datum['statement'])
doc.vector.shape
statistic_types = {'CARDINAL', 'PERCENT', 'MONEY', 'QUANTITY'}

def stat_counter(text):
    if not isinstance(text, str):
        return 0
    doc = nlp(text)
    counter = 0
    for ent in doc.ents: 
        if ent.label_ in statistic_types:
            counter += 1
    return counter

In [86]:
from rapidfuzz import fuzz
conservative_bigrams = pd.read_csv('top_conservative_bigrams.csv')['bigram']
liberal_bigrams = pd.read_csv('top_liberal_bigrams.csv')['bigram']
def match_counter(statement, bigram_list, threshold):
    stat = nlp(str(statement))
    word = [word.text.lower() for word in stat]
    bigram_coll = [''.join(word[i:i+2]) for i in range(len(word)-1)]
    matches = 0
    for bigram in bigram_coll:
        for check in bigram_list:
            if fuzz.ratio(bigram, check) >= threshold:
                matches += 1
                break

    return matches

for df in [df_train, df_test, df_val]:
    df['statistic_count'] = df['statement'].apply(stat_counter)
    df['conservative_bigram_count'] = df['statement'].apply(
        lambda x: match_counter(x, conservative_bigrams, threshold=70)
    )
    df['liberal_bigram_count'] = df['statement'].apply(
        lambda x: match_counter(x, liberal_bigrams, threshold=70)
    )

# Factuality Factor 4: Sensationalism (Ryan)

In [87]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def emotional_intensity_vader(text):
    if not isinstance(text, str):
        return 0.0
    if len(text) == 0:
        return 0.0
    vs = analyzer.polarity_scores(text)
    return abs(vs['compound'])

for df in [df_train, df_test, df_val]:
    df['emotional_intensity'] = df['statement'].apply(emotional_intensity_vader)

In [88]:
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, LabelEncoder
from sklearn.model_selection import cross_val_score, StratifiedKFold
from xgboost import XGBClassifier
import numpy as np
import pandas as pd


sentence_model = SentenceTransformer("all-MiniLM-L6-v2", device="mps")  # or 'cpu'/'cuda'


def embed_statement(text, chunk_size=200, overlap=50):
    words = str(text).split()
    if len(words) <= chunk_size:
        return sentence_model.encode([text])[0]
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size - overlap)]
    embeddings = sentence_model.encode(chunks)
    return np.mean(embeddings, axis=0)  

df_train["statement"] = df_train["statement"].fillna("")
df_train["embedding"] = df_train["statement"].apply(embed_statement)
num_cols = ['word_count', 'statistic_count', 'conservative_bigram_count','liberal_bigram_count', 'emotional_intensity','bert_pred_class',
            'bert_prob_class_0','bert_prob_class_1','bert_prob_class_2','bert_prob_class_3','bert_prob_class_4','bert_prob_class_5','spam_score']
# category_cols = ['party_affiliation', 'party_category', 'subject']
# category_cols = ['subject']

existing_num_cols = [c for c in num_cols if c in df_train.columns]
# existing_cat_cols = [c for c in category_cols if c in df_train.columns]
scaler = StandardScaler()
structured_df = pd.DataFrame(
    scaler.fit_transform(df_train[existing_num_cols]),
    columns=existing_num_cols,
    index=df_train.index
)
encoder = OrdinalEncoder()
# cat_df = pd.DataFrame(
#     encoder.fit_transform(df_train[existing_cat_cols].fillna("missing")),
#     columns=existing_cat_cols,
#     index=df_train.index
# )
embeddings = np.vstack(df_train["embedding"].values)
embedding_cols = [f"emb_{i}" for i in range(embeddings.shape[1])]
embedding_df = pd.DataFrame(embeddings, columns=embedding_cols, index=df_train.index)
X = pd.concat([structured_df, embedding_df], axis=1) # removed cat_df for now
le = LabelEncoder()
y = le.fit_transform(df_train["label"])
model = XGBClassifier(
    objective='multi:softprob',
    num_class=len(le.classes_),
    max_depth=6,
    n_estimators=500,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method='hist',
    enable_categorical=True,
    eval_metric='mlogloss'
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
print(f"Cross-validated accuracy: {scores.mean():.4f} ± {scores.std():.4f}")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/model_selection/_split.py:776: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


Cross-validated accuracy: 0.2637 ± 0.0057


In [90]:
# test_data = pd.read_csv('data/liar-plus/test2.tsv', sep='\t', header=None)
# test_data.columns = ['index','id', 'label', 'statement', 'subject', 'speaker', 'speaker_job_title', 'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context','justification']
# test_data.drop(columns=['index'], inplace=True)
party_map = {
    'republican': 'right-leaning',
    'democrat': 'left-leaning',
    'libertarian': 'right-leaning',
    'tea-party-member': 'right-leaning',
    'ocean-state-tea-party-action': 'right-leaning',
    'constitution-party': 'right-leaning',
    'democratic-farmer-labor': 'left-leaning',
    'green': 'left-leaning',
    'labor-leader': 'left-leaning',
    'liberal-party-canada': 'centrist',
    'Moderate': 'centrist',
    'independent': 'centrist',
    'none': 'other',
    'organization': 'other',
    'columnist': 'other',
    'activist': 'other',
    'talk-show-host': 'other',
    'newsmaker': 'other',
    'journalist': 'other',
    'state-official': 'other',
    'business-leader': 'other',
    'education-official': 'other',
    'government-body': 'other'
}

df_test['party_category'] = df_test['party_affiliation'].map(party_map).fillna('other')

columns_drop = [
    'speaker_job_title', 'state_info', 
    'barely_true_counts', 'false_counts', 'half_true_counts', 
    'mostly_true_counts', 'pants_on_fire_counts',
    'speaker', 'context', 'justification'
]
test_df = df_test.drop(columns=columns_drop)

test_df['word_count'] = test_df['statement'].apply(lambda x: len(x.split()) if pd.notnull(x) else 0)
test_df['statistic_count'] = test_df['statement'].apply(stat_counter)
test_df['conservative_bigram_count'] = test_df['statement'].apply(
    lambda x: match_counter(x, conservative_bigrams, threshold=70)
)
test_df['liberal_bigram_count'] = test_df['statement'].apply(
    lambda x: match_counter(x, liberal_bigrams, threshold=70)
)
test_df['emotional_intensity'] = test_df['statement'].apply(emotional_intensity_vader)

test_df["statement"] = test_df["statement"].fillna("")
test_df["embedding"] = test_df["statement"].apply(embed_statement)

existing_num_cols = [c for c in num_cols if c in test_df.columns]
# existing_cat_cols = [c for c in category_cols if c in test_df.columns]

test_structured_df = pd.DataFrame(
    scaler.transform(test_df[existing_num_cols]),
    columns=existing_num_cols,
    index=test_df.index
)

# test_cat_data = test_df[existing_cat_cols].fillna("missing").copy()

# test_cat_encoded = {}
# for i, col in enumerate(existing_cat_cols):
#     train_categories = encoder.categories_[i]
#     category_map = {cat: idx for idx, cat in enumerate(train_categories)}
#     test_cat_encoded[col] = test_cat_data[col].map(category_map)
#     max_train_value = len(train_categories) - 1
#     test_cat_encoded[col] = test_cat_encoded[col].fillna(max_train_value + 1).astype(int)

# test_cat_df = pd.DataFrame(test_cat_encoded, index=test_df.index)

test_embeddings = np.vstack(test_df["embedding"].values)
test_embedding_df = pd.DataFrame(test_embeddings, columns=embedding_cols, index=test_df.index)

X_test = pd.concat([test_structured_df, test_embedding_df], axis=1) # removed test_cat_df for now

# y_test_pred = model.predict(X_test)
# y_test_pred_proba = model.predict_proba(X_test)

# y_test_pred_labels = le.inverse_transform(y_test_pred)

# test_df['predicted_label'] = y_test_pred_labels
# test_df['predicted_probability'] = np.max(y_test_pred_proba, axis=1)

# true_labels = test_df['label']
# accuracy = np.mean(y_test_pred_labels == true_labels)
# print(f"Test Accuracy: ")
# print(accuracy)

# Muller Loop

In [91]:
# fit_cols = [
#     #'statement',
#     'bert_pred_class','bert_prob_class_0', 'bert_prob_class_1', 'bert_prob_class_2', 'bert_prob_class_3', 'bert_prob_class_4', 'bert_prob_class_5', 'bert_prob_class_6','spam_class']
# df_train.dropna(subset=fit_cols + ['label'], inplace=True)
# df_test.dropna(subset=fit_cols + ['label'], inplace=True)
# X_train = df_train[fit_cols].drop(columns=['bert_pred_class'])
# y_train = df_train['label']
# X_test = df_test[fit_cols].drop(columns=['bert_pred_class'])
y_test = le.transform(df_test['label'])

In [ ]:
# from time import time
# import numpy as np
# import matplotlib.pyplot as plt
# from matplotlib.colors import ListedColormap
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler
# from sklearn.datasets import make_moons, make_circles, make_classification
# from sklearn.neural_network import MLPClassifier
# from sklearn.neighbors import KNeighborsClassifier
# from sklearn.svm import SVC
# from sklearn.gaussian_process import GaussianProcessClassifier
# from sklearn.gaussian_process.kernels import RBF
# from sklearn.tree import DecisionTreeClassifier
# from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
# from sklearn.naive_bayes import GaussianNB
# from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

# names = ["Nearest Neighbors", "Linear SVM", "RBF SVM", #"Gaussian Process",
#          "Decision Tree", "Random Forest", "Neural Net", "AdaBoost",
#          "Naive Bayes", "QDA"]

# classifiers = [
#     KNeighborsClassifier(2),
#     SVC(kernel="linear", C=0.025),
#     SVC(gamma=2, C=1),
# #     GaussianProcessClassifier(1.0 * RBF(1.0)),
#     DecisionTreeClassifier(max_depth=5),
#     RandomForestClassifier(max_depth=5, n_estimators=10, max_features=1),
#     MLPClassifier(alpha=1, max_iter=1000),
#     AdaBoostClassifier(),
#     GaussianNB(),
#     QuadraticDiscriminantAnalysis()]

# # TODO (Apply): All cross-validation

# max_score = 0.0
# max_class = ''
# # iterate over classifiers
# for name, clf in zip(names, classifiers):
#     start_time = time()
#     clf.fit(X, y)
#     score = 100.0 * clf.score(X_test, y_test)
#     print('Classifier = %s, Score (test, accuracy) = %.2f,' %(name, score), 'Training time = %.2f seconds' % (time() - start_time))
    
#     if score > max_score:
#         clf_best = clf
#         max_score = score
#         max_class = name

# print(80*'-' )
# print('Best --> Classifier = %s, Score (test, accuracy) = %.2f' %(max_class, max_score))

In [92]:
from sklearn.decomposition import PCA

pca = PCA(n_components=50)
X_train_reduced = pca.fit_transform(X)
X_test_reduced = pca.transform(X_test)

In [103]:
X

,word_count,statistic_count,conservative_bigram_count,liberal_bigram_count,emotional_intensity,spam_score,emb_0,emb_1,emb_2,emb_3,...,emb_374,emb_375,emb_376,emb_377,emb_378,emb_379,emb_380,emb_381,emb_382,emb_383
0,-0.707854,-0.452703,-0.702534,-0.72217,-0.125039,0.209836,-0.023657,-0.086201,-0.044484,0.015858,...,0.061642,0.031746,-0.080949,-0.010309,0.062963,0.025709,0.084539,-0.060905,0.166828,0.039760
1,0.601618,-0.452703,0.298907,-0.72217,0.290155,-0.450672,0.014051,0.031802,0.055212,0.047837,...,0.061461,-0.002610,0.044167,-0.001004,-0.078507,-0.079317,-0.010799,0.017237,-0.061281,-0.001477
2,0.097975,-0.452703,-0.702534,-0.72217,0.129603,-0.407042,-0.016569,0.012517,-0.035007,0.026673,...,0.037187,0.086657,-0.045857,-0.075116,-0.039440,0.003276,0.005033,0.050504,0.071088,-0.070123
3,-0.607126,-0.452703,0.298907,1.15059,1.771339,0.207987,-0.074179,0.063911,0.033571,0.008439,...,0.002507,0.070096,0.038114,-0.030645,-0.006283,0.054631,0.021744,-0.128516,0.086446,-0.095068
4,-0.808583,-0.452703,-0.702534,1.15059,-1.058480,-0.466174,0.009059,-0.009239,0.056488,0.028030,...,0.099117,-0.077822,0.015376,0.015565,-0.135522,0.028486,-0.001721,-0.096078,-0.027198,0.095355
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10237,-0.103482,-0.452703,-0.702534,0.21421,1.744083,0.021091,0.004386,-0.032098,0.047506,0.048836,...,-0.013047,0.027880,0.039358,-0.039163,0.040587,-0.012937,0.055977,-0.027015,0.017876,0.047906
10238,-0.405668,-0.452703,0.298907,1.15059,0.442119,-0.406301,-0.023789,0.030235,-0.008846,0.037107,...,0.039730,-0.128908,-0.005140,0.023496,0.041325,-0.030299,0.016746,0.009866,0.040038,-0.015803
10239,1.004533,-0.452703,0.298907,-0.72217,1.129132,0.195934,-0.001566,0.052812,-0.041570,0.039299,...,-0.050827,-0.023327,0.051421,0.041582,-0.057477,-0.028695,0.024356,-0.076488,0.029235,0.088331
10240,-0.707854,-0.452703,-0.702534,-0.72217,-1.058480,-0.438028,-0.079076,0.090826,-0.003922,0.035713,...,0.070683,0.067400,-0.001681,0.003638,0.058003,-0.008409,0.055646,-0.077675,0.003842,-0.037684


In [102]:
from sklearn.metrics import classification_report

rf = RandomForestClassifier(max_depth=250, n_estimators=100, max_features=1)
rf.fit(X, y)
y_test_pred = rf.predict(X_test)

print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

           0       0.20      0.12      0.15       212
           1       0.27      0.37      0.31       249
           2       0.21      0.32      0.26       265
           3       0.21      0.26      0.23       241
           4       1.00      0.02      0.04        92
           5       0.23      0.11      0.14       208

    accuracy                           0.23      1267
   macro avg       0.35      0.20      0.19      1267
weighted avg       0.28      0.23      0.21      1267



In [ ]:
X.to_parquet('train_features.parquet')
X_test.to_parquet('test_features.parquet')
# y_test.to_parquet('test_labels.parquet')
# y.to_parquet('train_labels.parquet')

In [99]:
def bert_pred(df_in):
    MODEL_NAME = "bert-base-uncased"   # swap if you prefer another checkpoint
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=6)
    batch_size = 32
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
    model.to(device)
    df = df_in.copy()
    # Ensure statements are strings
    df['statement'] = df['statement'].astype(str)

    # Tokenize this specific df
    encodings = tokenizer(
        df['statement'].tolist(),
        truncation=True,
        padding=True,
        return_tensors='pt'
    )

    input_ids = encodings['input_ids'].to(device)
    attention_mask = encodings['attention_mask'].to(device)

    # Batch prediction
    all_probs = []
    for i in range(0, len(df), batch_size):
        batch_ids = input_ids[i:i+batch_size]
        batch_mask = attention_mask[i:i+batch_size]
        with torch.no_grad():
            batch_outputs = model(batch_ids, attention_mask=batch_mask)
            batch_probs = torch.softmax(batch_outputs.logits, dim=1)
            all_probs.append(batch_probs.cpu())

    probs = torch.cat(all_probs, dim=0)

    # Predicted class
    df['bert_pred_class'] = torch.argmax(probs, dim=1).numpy()

    # Probabilities per class
    num_labels = probs.shape[1]
    for j in range(num_labels):
        df[f'bert_prob_class_{j}'] = probs[:, j].numpy()
    return df

In [100]:
def prep_df(article_df):
    out_df = bert_pred(article_df)
    out_df['spam_score'] = get_spam_scores(out_df['statement'].tolist())
    out_df['word_count'] = out_df['statement'].apply(lambda x: len(x.split()) if pd.notnull(x) else 0)
    out_df['statistic_count'] = out_df['statement'].apply(stat_counter)
    out_df['conservative_bigram_count'] = out_df['statement'].apply(
        lambda x: match_counter(x, conservative_bigrams, threshold=70)
    )
    out_df['liberal_bigram_count'] = out_df['statement'].apply(
        lambda x: match_counter(x,liberal_bigrams, threshold=70)
    )
    out_df['emotional_intensity'] = out_df['statement'].apply(emotional_intensity_vader)
    out_df["statement"] = out_df["statement"].fillna("")
    out_df["embedding"] = out_df["statement"].apply(embed_statement)
    num_cols = ['word_count', 'statistic_count', 'conservative_bigram_count','liberal_bigram_count', 'emotional_intensity','bert_pred_class','bert_prob_class_0','bert_prob_class_1','bert_prob_class_2','bert_prob_class_3','bert_prob_class_4','bert_prob_class_5','spam_score']
    category_cols = ['subject']

    existing_num_cols = [c for c in num_cols if c in out_df.columns]
    existing_cat_cols = [c for c in category_cols if c in out_df.columns]
    scaler = StandardScaler()
    structured_df = pd.DataFrame(
        scaler.fit_transform(out_df[existing_num_cols]),
        columns=existing_num_cols,
        index=out_df.index
    )
    encoder = OrdinalEncoder()
    cat_df = pd.DataFrame(
        encoder.fit_transform(out_df[existing_cat_cols].fillna("missing")),
        columns=existing_cat_cols,
        index=out_df.index
    )
    embeddings = np.vstack(out_df["embedding"].values)
    embedding_cols = [f"emb_{i}" for i in range(embeddings.shape[1])]
    embedding_df = pd.DataFrame(embeddings, columns=embedding_cols, index=out_df.index)
    X = pd.concat([structured_df, cat_df, embedding_df], axis=1)
    return X

article_df = pd.read_parquet('clean_articles.parquet')
outside_df = prep_df(article_df)
rf.predict(outside_df)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- bert_pred_class
- bert_prob_class_0
- bert_prob_class_1
- bert_prob_class_2
- bert_prob_class_3
- ...


In [ ]:
outside_df['predicted_label'] = rf.predict(outside_df)
outside_df['prediction'] = le.inverse_transform(outside_df['predicted_label'])
outside_df

,word_count,statistic_count,conservative_bigram_count,liberal_bigram_count,emotional_intensity,bert_pred_class,bert_prob_class_0,bert_prob_class_1,bert_prob_class_2,bert_prob_class_3,...,emb_376,emb_377,emb_378,emb_379,emb_380,emb_381,emb_382,emb_383,predicted_label,prediction
0,1.594180,1.016085,1.780321,0.788608,0.574078,-2.449490,-0.052119,-0.808516,1.530699,0.921167,...,-0.014489,-0.049566,-0.003445,0.046242,0.001997,-0.064617,0.002740,0.020356,0,barely-true
1,-0.441415,-0.714006,-0.520401,-0.620819,0.686191,0.408248,-1.060913,1.035584,1.010302,0.621100,...,0.044327,-0.051209,-0.032942,-0.005285,0.057363,0.000655,0.035130,0.030683,5,true
2,0.813716,1.592782,0.725823,0.671156,-2.130645,0.408248,-0.426649,0.249303,-0.435886,-0.461663,...,0.019836,-0.040687,0.014086,-0.043908,0.009465,0.003888,-0.045349,0.053538,2,half-true
3,-0.970852,-0.329541,-0.999719,-1.208081,-0.687192,0.408248,0.976993,-1.615336,0.710102,0.249487,...,0.028794,0.027949,-0.015948,0.023987,0.034614,-0.050473,-0.006657,0.026848,0,barely-true
4,0.891306,0.631620,0.725823,1.728227,0.833339,0.408248,-1.493817,-0.640349,-0.828779,-1.953549,...,-0.058912,0.014098,0.030293,-0.007878,0.063113,-0.059762,0.034936,-0.002444,3,mostly-true
5,-0.738082,-1.098470,-0.712128,-0.385915,0.701957,0.408248,0.587170,0.337672,-0.617611,-0.548388,...,-0.050299,0.002773,-0.036174,0.019732,-0.021176,-0.014232,0.070258,0.027992,2,half-true
6,-1.148853,-1.098470,-0.999719,-0.973176,0.022272,0.408248,1.469335,1.441643,-1.368828,1.171847,...,0.014955,-0.064711,-0.044999,0.012829,0.014632,-0.022534,-0.020175,0.079156,2,half-true


In [ ]:
outside_df['name'] = np.array(['bill gates carpet bomb', 'toronto99: liberals election rigging', 'babylon bee waterslide', 'the onion: diplomatic halloween decoration', 'NPR Lebanon Cancer', 'MSNBC obergefeld book', 'NYT Trump tearing down east wing'])

In [ ]:
outside_df

,word_count,statistic_count,conservative_bigram_count,liberal_bigram_count,emotional_intensity,bert_pred_class,bert_prob_class_0,bert_prob_class_1,bert_prob_class_2,bert_prob_class_3,...,emb_377,emb_378,emb_379,emb_380,emb_381,emb_382,emb_383,predicted_label,prediction,name
0,1.594180,1.016085,1.780321,0.788608,0.574078,-2.449490,-0.052119,-0.808516,1.530699,0.921167,...,-0.049566,-0.003445,0.046242,0.001997,-0.064617,0.002740,0.020356,0,barely-true,bill gates carpet bomb
1,-0.441415,-0.714006,-0.520401,-0.620819,0.686191,0.408248,-1.060913,1.035584,1.010302,0.621100,...,-0.051209,-0.032942,-0.005285,0.057363,0.000655,0.035130,0.030683,5,true,toronto99: liberals election rigging
2,0.813716,1.592782,0.725823,0.671156,-2.130645,0.408248,-0.426649,0.249303,-0.435886,-0.461663,...,-0.040687,0.014086,-0.043908,0.009465,0.003888,-0.045349,0.053538,2,half-true,babylon bee waterslide
3,-0.970852,-0.329541,-0.999719,-1.208081,-0.687192,0.408248,0.976993,-1.615336,0.710102,0.249487,...,0.027949,-0.015948,0.023987,0.034614,-0.050473,-0.006657,0.026848,0,barely-true,the onion: diplomatic halloween decoration
4,0.891306,0.631620,0.725823,1.728227,0.833339,0.408248,-1.493817,-0.640349,-0.828779,-1.953549,...,0.014098,0.030293,-0.007878,0.063113,-0.059762,0.034936,-0.002444,3,mostly-true,NPR Lebanon Cancer
5,-0.738082,-1.098470,-0.712128,-0.385915,0.701957,0.408248,0.587170,0.337672,-0.617611,-0.548388,...,0.002773,-0.036174,0.019732,-0.021176,-0.014232,0.070258,0.027992,2,half-true,MSNBC obergefeld book
6,-1.148853,-1.098470,-0.999719,-0.973176,0.022272,0.408248,1.469335,1.441643,-1.368828,1.171847,...,-0.064711,-0.044999,0.012829,0.014632,-0.022534,-0.020175,0.079156,2,half-true,NYT Trump tearing down east wing


Why model failed: many things to fine tune
* BERT transformer may need more fine tuning
* model selection was slowed by muller loop, will try our own models
* tricky wording in satirical articles
* lack of extensive text-preprocessing
* otherworldly events

# Predict article truthfulness

In [ ]:
import requests
from bs4 import BeautifulSoup

def scrape_article_text(url):
    try:
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        article_text = ""
        article_content = soup.find('article') or soup.find('div', class_='article-content') or soup.find('div', class_='entry-content')
        
        if article_content:
            paragraphs = article_content.find_all('p')
            article_text = ' '.join([p.get_text().strip() for p in paragraphs])
        else:
            paragraphs = soup.find_all('p')
            article_text = ' '.join([p.get_text().strip() for p in paragraphs if len(p.get_text()) > 50])
        
        return article_text[:2000]
    except Exception as e:
        print(f"Error scraping article: {e}")
        return ""

def predict_article_truthfulness(url):
    article_text = scrape_article_text(url)
    
    if not article_text:
        print("Could not extract article text")
        return
    
    article_data = {
        'statement': article_text,
        'subject': 'history',
        'party_affiliation': 'republican',
        'party_category': 'right-leaning',
        'word_count': len(article_text.split()),
        'statistic_count': stat_counter(article_text),
        'conservative_bigram_count': match_counter(article_text, conservative_bigrams, threshold=70),
        'liberal_bigram_count': match_counter(article_text, liberal_bigrams, threshold=70),
        'emotional_intensity': emotional_intensity_vader(article_text)
    }
    
    article_df = pd.DataFrame([article_data])
    article_df["embedding"] = article_df["statement"].apply(embed_statement)
    
    article_structured = pd.DataFrame(
        scaler.transform(article_df[existing_num_cols]),
        columns=existing_num_cols,
        index=article_df.index
    )
    
    article_cat_data = article_df[existing_cat_cols].fillna("missing").copy()
    article_cat_encoded = {}
    
    for i, col in enumerate(existing_cat_cols):
        train_categories = encoder.categories_[i]
        category_map = {cat: idx for idx, cat in enumerate(train_categories)}
        article_cat_encoded[col] = article_cat_data[col].map(category_map)
        max_train_value = len(train_categories) - 1
        article_cat_encoded[col] = article_cat_encoded[col].fillna(max_train_value + 1).astype(int)
    
    article_cat_df = pd.DataFrame(article_cat_encoded, index=article_df.index)
    article_embeddings = np.vstack(article_df["embedding"].values)
    article_embedding_df = pd.DataFrame(article_embeddings, columns=embedding_cols, index=article_df.index)
    
    X_article = pd.concat([article_structured, article_cat_df, article_embedding_df], axis=1)
    
    prediction = clf_best.predict(X_article)[0]
    prediction_proba = clf_best.predict_proba(X_article)[0]
    predicted_label = le.inverse_transform([prediction])[0]
    
    confidence = np.max(prediction_proba)
    
    print("Article Prediction")
    print(predicted_label)
   
    
    return predicted_label, confidence

url = "https://dailycaller.com/2025/10/10/ingersoll-truth-about-christopher-columbus-day/"
prediction, confidence = predict_article_truthfulness(url)

KeyError: "['bert_pred_class', 'bert_prob_class_0', 'bert_prob_class_1', 'bert_prob_class_2', 'bert_prob_class_3', 'bert_prob_class_4', 'bert_prob_class_5', 'spam_score'] not in index"